# LIWC merge

Merges the LIWC and DeLIWC result files in this folder into one long table.

Output columns: `subject_id`, `language`, `lexicon`, `aggregation`, `metric`, `value`.
All values are `aggregation = total`, since the LIWC outputs are one score per subject.

In [7]:
from pathlib import Path
import pandas as pd
import operator

def count_words(text):
    if pd.isna(text):
        return 0
    return sum(len(word) >= 2 for word in str(text).split(" "))

HERE = Path.cwd()

files = [("LIWC22_english.csv", "liwc_2022_results.xlsx"), 
         ("LIWC15_english.csv", "liwc_2015_results.xlsx"),
         ("LIWC15_german.csv", "deliwc_2015_results.xlsx")]

for (infile, outfile) in files:
    # Input and output files
    input_file = HERE / infile
    output_file = HERE / outfile

    # Read CSV
    df = pd.read_csv(input_file, encoding="utf-8-sig")

    # Remove accidental spaces from column names
    df.columns = df.columns.str.strip()

    df["word_count_english"] = df["english_text"].apply(count_words)
    df["word_count_german"] = df["german_text"].apply(count_words)

    df = df.rename(columns={
        "emo_pos": "posemo", 
        "emo_neg": "negemo",
        "tone_pos": "postone",
        "tone_neg": "negtone"
    })
    # Export to Excel without row index
    # Keep only the required columns
    required_cols = [
        "subject_id", "postone", "negtone", "posemo", "negemo",
        "word_count_english", "word_count_german"]
    df_filtered = df[[col for col in required_cols if col in df.columns]].copy()
    df_filtered.to_excel(output_file, index=False)

    print(f"Saved {len(df_filtered)} rows to {output_file}")


Saved 5 rows to /Users/katra/Library/Mobile Documents/com~apple~CloudDocs/sentiment_paper/data/liwc_results/liwc_2022_results.xlsx
Saved 5 rows to /Users/katra/Library/Mobile Documents/com~apple~CloudDocs/sentiment_paper/data/liwc_results/liwc_2015_results.xlsx
Saved 5 rows to /Users/katra/Library/Mobile Documents/com~apple~CloudDocs/sentiment_paper/data/liwc_results/deliwc_2015_results.xlsx


In [8]:
# file -> (language, lexicon)
FILES = {
    "deliwc_2015_results.xlsx": ("german", "deliwc_2015"),
    "liwc_2022_results.xlsx":   ("english", "liwc_2022"),
    "liwc_2015_results.xlsx":   ("english", "liwc_2015"),
}

DERIVED = {
    "emotion_frequency":      ("posemo", operator.add, "negemo"),
    "emotion_polarity_index": ("posemo", operator.sub, "negemo"),
    "tone_frequency":         ("postone", operator.add, "negtone"),
    "tone_polarity_index":    ("postone", operator.sub, "negtone"),
}
OUT_PATH = HERE / "liwc_long.xlsx"


In [9]:
for file_name, (language, lexicon) in FILES.items():
    path = HERE / file_name
    if not path.exists():
            print(f"Skipping {file_name}: not found")
            continue
    
    wide = pd.read_excel(path)
    print(wide.columns)
    

Index(['subject_id', 'posemo', 'negemo', 'word_count_english',
       'word_count_german'],
      dtype='str')
Index(['subject_id', 'postone', 'negtone', 'posemo', 'negemo',
       'word_count_english', 'word_count_german'],
      dtype='str')
Index(['subject_id', 'posemo', 'negemo', 'word_count_english',
       'word_count_german'],
      dtype='str')


In [13]:
frames = []

for file_name, (language, lexicon) in FILES.items():
    path = HERE / file_name

    if not path.exists():
        print(f"Skipping {file_name}: not found")
        continue

    wide = pd.read_excel(path)

    # 1. Calculate derived metrics
    for metric, (col1, operation, col2) in DERIVED.items():
        if col1 in wide.columns and col2 in wide.columns:
            wide[metric] = operation(wide[col1], wide[col2])

    # 2. Choose word count based on language
    word_count_col = f"word_count_{language}"

    # 3. Calculate emotion_count
    if all(c in wide.columns for c in ["posemo", "negemo", word_count_col]):
        wide["emotion_count"] = (
            (wide["posemo"] + wide["negemo"])
            * wide[word_count_col]
        )

    # 4. Calculate tone_count (English only)
    if language == "english" and all(
        c in wide.columns for c in ["postone", "negtone", word_count_col]
    ):
        wide["tone_count"] = (
            (wide["postone"] + wide["negtone"])
            * wide[word_count_col]
        )

    # 5. Melt to long format
    long = wide.melt(
        id_vars="subject_id",
        value_vars=[
            c for c in wide.columns
            if c != "subject_id" and not c.startswith("word_count_")
        ],
        var_name="metric",
        value_name="value"
    )

    # 6. Add metadata
    long["language"] = language
    long["lexicon"] = lexicon
    long["aggregation"] = "total"  # Default aggregation type

    frames.append(long)

    print(f"{file_name}: {wide.shape[0]} subjects")
    print(f"Metrics: {list(long['metric'].unique())}")

# 7. Merge all files
merged = pd.concat(frames, ignore_index=True)

# 8. Set final column order
merged = merged[
    ["subject_id", "metric", "value", "language", "lexicon", "aggregation"]
]

display(merged)

merged.to_excel(OUT_PATH, index=False, engine="openpyxl")
print("Wrote", OUT_PATH)

deliwc_2015_results.xlsx: 5 subjects
Metrics: ['posemo', 'negemo', 'emotion_frequency', 'emotion_polarity_index', 'emotion_count']
liwc_2022_results.xlsx: 5 subjects
Metrics: ['postone', 'negtone', 'posemo', 'negemo', 'emotion_frequency', 'emotion_polarity_index', 'tone_frequency', 'tone_polarity_index', 'emotion_count', 'tone_count']
liwc_2015_results.xlsx: 5 subjects
Metrics: ['posemo', 'negemo', 'emotion_frequency', 'emotion_polarity_index', 'emotion_count']


,subject_id,metric,value,language,lexicon,aggregation
0,T0001,posemo,0.7268,german,deliwc_2015,total
1,T0002,posemo,0.9794,german,deliwc_2015,total
2,T0003,posemo,0.6400,german,deliwc_2015,total
3,T0004,posemo,0.6536,german,deliwc_2015,total
4,T0005,posemo,0.9282,german,deliwc_2015,total
...,...,...,...,...,...,...
95,T0001,emotion_count,431.9720,english,liwc_2015,total
96,T0002,emotion_count,242.0236,english,liwc_2015,total
97,T0003,emotion_count,331.8770,english,liwc_2015,total
98,T0004,emotion_count,255.5882,english,liwc_2015,total


Wrote /Users/katra/Library/Mobile Documents/com~apple~CloudDocs/sentiment_paper/data/liwc_results/liwc_long.xlsx


In [16]:
# Check: one row per subject x lexicon x metric
dupes = merged.duplicated(["subject_id", "lexicon", "aggregation", "metric"]).sum()
print("Rows:", len(merged), "| duplicate keys:", dupes)
merged.groupby(["lexicon", "language"])["metric"].unique()
print("Metrics:", merged["metric"].unique())
print("Lexicons:", merged["lexicon"].unique())


Rows: 100 | duplicate keys: 0
Metrics: <StringArray>
[                'posemo',                 'negemo',      'emotion_frequency',
 'emotion_polarity_index',          'emotion_count',                'postone',
                'negtone',         'tone_frequency',    'tone_polarity_index',
             'tone_count']
Length: 10, dtype: str
Lexicons: <StringArray>
['deliwc_2015', 'liwc_2022', 'liwc_2015']
Length: 3, dtype: str
